**1. Analise exploratoria inicial**

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
pd.set_option('display.max_rows', 50)
print('Setup Complete')

/kaggle/input/datasets/imdevskp/corona-virus-report/covid_19_clean_complete.csv
/kaggle/input/datasets/imdevskp/corona-virus-report/country_wise_latest.csv
/kaggle/input/datasets/imdevskp/corona-virus-report/day_wise.csv
/kaggle/input/datasets/imdevskp/corona-virus-report/usa_county_wise.csv
/kaggle/input/datasets/imdevskp/corona-virus-report/worldometer_data.csv
/kaggle/input/datasets/imdevskp/corona-virus-report/full_grouped.csv
Setup Complete


In [3]:
covid_filepath='/kaggle/input/datasets/imdevskp/corona-virus-report/covid_19_clean_complete.csv'
covid_data=pd.read_csv(covid_filepath)

In [4]:
print(covid_data.shape) #dimensao da base de dados

(49068, 10)


In [5]:
covid_data.columns #nomes das colunas

Index(['Province/State', 'Country/Region', 'Lat', 'Long', 'Date', 'Confirmed',
       'Deaths', 'Recovered', 'Active', 'WHO Region'],
      dtype='object')

In [6]:
print(covid_data.dtypes) #mostra os tipos dos dados 

Province/State     object
Country/Region     object
Lat               float64
Long              float64
Date               object
Confirmed           int64
Deaths              int64
Recovered           int64
Active              int64
WHO Region         object
dtype: object


In [8]:
covid_data['Date']=pd.to_datetime(covid_data['Date']) #troca tipo de dado
###print(covid_data.dtypes) #mostra os tipos dos dados apos modificacao

In [9]:
covid_data.describe() #resumo estatistico dos dados

,Lat,Long,Date,Confirmed,Deaths,Recovered,Active
count,49068.000000,49068.000000,49068,4.906800e+04,49068.000000,4.906800e+04,4.906800e+04
mean,21.433730,23.528236,2020-04-24 12:00:00,1.688490e+04,884.179160,7.915713e+03,8.085012e+03
min,-51.796300,-135.000000,2020-01-22 00:00:00,0.000000e+00,0.000000,0.000000e+00,-1.400000e+01
25%,7.873054,-15.310100,2020-03-08 18:00:00,4.000000e+00,0.000000,0.000000e+00,0.000000e+00
50%,23.634500,21.745300,2020-04-24 12:00:00,1.680000e+02,2.000000,2.900000e+01,2.600000e+01
75%,41.204380,80.771797,2020-06-10 06:00:00,1.518250e+03,30.000000,6.660000e+02,6.060000e+02
max,71.706900,178.065000,2020-07-27 00:00:00,4.290259e+06,148011.000000,1.846641e+06,2.816444e+06
std,24.950320,70.442740,NaN,1.273002e+05,6313.584411,5.480092e+04,7.625890e+04


In [10]:
covid_data.isnull().sum()

Province/State    34404
Country/Region        0
Lat                   0
Long                  0
Date                  0
Confirmed             0
Deaths                0
Recovered             0
Active                0
WHO Region            0
dtype: int64

**2. Analise por provincias da China**

In [13]:
china_data= covid_data[covid_data['Country/Region']=='China'] #dataframe exclusivamente chines (2.1 (b))
china_data['Province/State'].unique() #listar provincias chinesas presentes (2.1 (a))

array(['Anhui', 'Beijing', 'Chongqing', 'Fujian', 'Gansu', 'Guangdong',
       'Guangxi', 'Guizhou', 'Hainan', 'Hebei', 'Heilongjiang', 'Henan',
       'Hong Kong', 'Hubei', 'Hunan', 'Inner Mongolia', 'Jiangsu',
       'Jiangxi', 'Jilin', 'Liaoning', 'Macau', 'Ningxia', 'Qinghai',
       'Shaanxi', 'Shandong', 'Shanghai', 'Shanxi', 'Sichuan', 'Tianjin',
       'Tibet', 'Xinjiang', 'Yunnan', 'Zhejiang'], dtype=object)

In [14]:
infos_china=['Province/State','Confirmed','Active','Deaths','Recovered']
clean_china=china_data[infos_china] #selecionadas apenas as colunas desejadas (2.1 (c))
clean_china

,Province/State,Confirmed,Active,Deaths,Recovered
48,Anhui,1,1,0,0
49,Beijing,14,14,0,0
50,Chongqing,6,6,0,0
51,Fujian,1,1,0,0
52,Gansu,0,0,0,0
...,...,...,...,...,...
48883,Tianjin,204,6,3,195
48884,Tibet,1,0,0,1
48885,Xinjiang,311,235,3,73
48886,Yunnan,190,2,2,186


In [16]:
china_grouped=clean_china.groupby('Province/State').sum() #agrupado os dados por provincia (2.1 (d))

In [17]:
china_5=china_grouped.nlargest(5, 'Confirmed')
china_5 #as cinco provincias com mais casos confirmados (2.1 (e))

,Confirmed,Active,Deaths,Recovered
Province/State,,,,
Hubei,11473248,1433798,651932,9387518
Guangdong,268051,29668,1273,237110
Henan,222581,218953,3628,0
Zhejiang,220824,21888,159,198777
Hunan,178641,15540,662,162439


**3. Tratamento de dados**

In [20]:
linha_com_provincia=covid_data.iloc[54] #para testar na funcao
linha_sem_provincia=covid_data.iloc[1]

criada duas variaveis, uma sem valor determinado para provincia e outra com, para determinar se a funcao estava de fato funcionando

In [25]:
def country_and_province(row): #definida a funcao que concatena valores
                               #caso o dado da provincia exista (3.(a))
    if pd.notna(row['Province/State']):
        return f'{row['Country/Region']}-{row['Province/State']}'
        #uso da string formatada para poder juntar duas variaveis e um texto
    else:
        return row['Country/Region']
###print(country_and_province(linha_com_provincia))
###print(country_and_province(linha_sem_provincia))

In [26]:
covid_data['Province/State'].notna()
#verificar se existe dado sobre a provincia (3(b))

0        False
1        False
2        False
3        False
4        False
         ...  
49063    False
49064    False
49065    False
49066    False
49067    False
Name: Province/State, Length: 49068, dtype: bool

In [69]:
covid_copy=covid_data.copy()
#cria uma copia do df original (3(c))

In [28]:
covid_copy['Country-Province']=covid_copy.apply(country_and_province, axis=1) 
#cria uma nova coluna chamada "Contry-Province"
#axis=1 pois e horizontal
covid_copy.head() #aplicada a funcao e chamado o .head() para 
                  #analisar se a funcao foi aplicada corretamente
                  # (3(d))


,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region,Country-Province
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean,Afghanistan
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe,Albania
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa,Algeria
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe,Andorra
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa,Angola


In [29]:
covid_copy.iloc[54] #usando a mesma linha que foi usada para mostrar 
                    #que a funcao funcionou no df copiado 

Province/State                  Guangxi
Country/Region                    China
Lat                             23.8298
Long                           108.7881
Date                2020-01-22 00:00:00
Confirmed                             2
Deaths                                0
Recovered                             0
Active                                2
WHO Region              Western Pacific
Country-Province          China-Guangxi
Name: 54, dtype: object

In [30]:
covid2=covid_copy.copy()
infos=['Country-Province','Lat','Long','Date','Confirmed',
       'Deaths','Recovered','Active','WHO Region', 'Country/Region']
covid_copy=covid_copy[infos]
covid_copy
#removida a coluna "Province/State" da copia do df
#e chamada a copia para avaliar (3(e))

,Country-Province,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region,Country/Region
0,Afghanistan,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean,Afghanistan
1,Albania,41.153300,20.168300,2020-01-22,0,0,0,0,Europe,Albania
2,Algeria,28.033900,1.659600,2020-01-22,0,0,0,0,Africa,Algeria
3,Andorra,42.506300,1.521800,2020-01-22,0,0,0,0,Europe,Andorra
4,Angola,-11.202700,17.873900,2020-01-22,0,0,0,0,Africa,Angola
...,...,...,...,...,...,...,...,...,...,...
49063,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa,Sao Tome and Principe
49064,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean,Yemen
49065,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa,Comoros
49066,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe,Tajikistan


**4. Ranking de mortes por milhao de habitantes por continente**

In [66]:
world_filepath='/kaggle/input/datasets/imdevskp/corona-virus-report/worldometer_data.csv'
world_data=pd.read_csv(world_filepath)
covid_copy.head()
#importado o arquivo da df (4(a))

,Country-Province,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region,Country/Region
0,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean,Afghanistan
1,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe,Albania
2,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa,Algeria
3,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe,Andorra
4,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa,Angola


In [41]:
infos=['Country/Region','Population','Continent']
world_data=world_data[infos]
world_data.head(50)
#selecionados apenas os dados de pais, populacao
#e continente na df "worldometer" (4(b))

,Country/Region,Population,Continent
0,USA,3.311981e+08,North America
1,Brazil,2.127107e+08,South America
2,India,1.381345e+09,Asia
3,Russia,1.459409e+08,Europe
4,South Africa,5.938157e+07,Africa
5,Mexico,1.290662e+08,North America
6,Peru,3.301632e+07,South America
7,Chile,1.913251e+07,South America
8,Colombia,5.093626e+07,South America
9,Spain,4.675665e+07,Europe


In [71]:
merged=pd.merge(world_data, covid_copy, on='Country/Region', how='inner')
merged=merged.sort_values(by=['Country/Region', 'Date'], ascending=True)
#ordena por pais usando as datas mais novas primeiro

#agrupado os dados das duas df (4(c))
merged #chamando a df para avaliar os dados 

,Country/Region,Continent,Population,TotalCases,NewCases,TotalDeaths,NewDeaths,TotalRecovered,NewRecovered,ActiveCases,...,WHO Region_x,Province/State,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region_y
13724,Afghanistan,Asia,39009447.0,36896,NaN,1298.0,NaN,25840.0,NaN,9758.0,...,EasternMediterranean,NaN,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
13725,Afghanistan,Asia,39009447.0,36896,NaN,1298.0,NaN,25840.0,NaN,9758.0,...,EasternMediterranean,NaN,33.939110,67.709953,2020-01-23,0,0,0,0,Eastern Mediterranean
13726,Afghanistan,Asia,39009447.0,36896,NaN,1298.0,NaN,25840.0,NaN,9758.0,...,EasternMediterranean,NaN,33.939110,67.709953,2020-01-24,0,0,0,0,Eastern Mediterranean
13727,Afghanistan,Asia,39009447.0,36896,NaN,1298.0,NaN,25840.0,NaN,9758.0,...,EasternMediterranean,NaN,33.939110,67.709953,2020-01-25,0,0,0,0,Eastern Mediterranean
13728,Afghanistan,Asia,39009447.0,36896,NaN,1298.0,NaN,25840.0,NaN,9758.0,...,EasternMediterranean,NaN,33.939110,67.709953,2020-01-26,0,0,0,0,Eastern Mediterranean
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24247,Zimbabwe,Africa,14883803.0,4339,NaN,84.0,NaN,1264.0,NaN,2991.0,...,Africa,NaN,-19.015438,29.154857,2020-07-23,2124,28,510,1586,Africa
24248,Zimbabwe,Africa,14883803.0,4339,NaN,84.0,NaN,1264.0,NaN,2991.0,...,Africa,NaN,-19.015438,29.154857,2020-07-24,2296,32,514,1750,Africa
24249,Zimbabwe,Africa,14883803.0,4339,NaN,84.0,NaN,1264.0,NaN,2991.0,...,Africa,NaN,-19.015438,29.154857,2020-07-25,2434,34,518,1882,Africa
24250,Zimbabwe,Africa,14883803.0,4339,NaN,84.0,NaN,1264.0,NaN,2991.0,...,Africa,NaN,-19.015438,29.154857,2020-07-26,2512,34,518,1960,Africa


In [72]:
merged=merged.groupby(['Country/Region','Continent','Date']).agg({
    'Population': 'mean',
    'Deaths': 'sum'
}).reset_index()
#junta os itens de mesmo pais e mesma data e agrega na linha
#define a populacao como a media da populacao das diferentes linhas,
#e soma as mortes, para que nao se percam dados de provincias diferentes
merged=merged.drop_duplicates(subset=['Country/Region'], keep='last')
merged.head(50)

,Country/Region,Continent,Date,Population,Deaths
187,Afghanistan,Asia,2020-07-27,39009447.0,1269
375,Albania,Europe,2020-07-27,2877470.0,144
563,Algeria,Africa,2020-07-27,43926079.0,1163
751,Andorra,Europe,2020-07-27,77278.0,52
939,Angola,Africa,2020-07-27,32956300.0,41
1127,Antigua and Barbuda,North America,2020-07-27,98010.0,3
1315,Argentina,South America,2020-07-27,45236884.0,3059
1503,Armenia,Asia,2020-07-27,2963811.0,711
1691,Australia,Australia/Oceania,2020-07-27,25528864.0,167
1879,Austria,Europe,2020-07-27,9011577.0,713


In [73]:
infos=['Country/Region','Population','Continent','Deaths']
merged=merged[infos] #mantido apenas as colunas pedidas (4(d))
merged.tail(50)
#data_final['Deaths per Million']=data_final['Deaths']/data_final['Population']*(10**6)
#data_final=data_final.sort_values(by='Deaths per Million', ascending=False)

,Country/Region,Population,Continent,Deaths
22935,Papua New Guinea,8963009.0,Australia/Oceania,0
23123,Paraguay,7141091.0,South America,43
23311,Peru,33016319.0,South America,18418
23499,Philippines,109722719.0,Asia,1945
23687,Poland,37842302.0,Europe,1676
23875,Portugal,10193593.0,Europe,1719
24063,Qatar,2807805.0,Asia,165
24251,Romania,19224023.0,Europe,2206
24439,Russia,145940924.0,Europe,13334
24627,Rwanda,12981546.0,Africa,5


In [74]:
merged=merged.drop_duplicates(subset=['Country/Region'], keep='last')
infos=['Continent','Population','Deaths']
merged=merged[infos]
merged=merged.groupby('Continent').sum()
#agrupado as linhas da df por continente
#em ordem alfabetica (4(e))
merged.head(50)

,Population,Deaths
Continent,,
Africa,1.215738e+09,17759
Asia,3.020510e+09,85740
Australia/Oceania,4.039107e+07,189
Europe,6.794130e+08,155775
North America,2.568683e+08,59215
South America,4.308076e+08,135506


In [76]:
merged['Deaths per million']=merged['Deaths']/merged['Population']*10**6
merged
#criada a nova coluna de mortes por milhao (4(f))

,Population,Deaths,Deaths per million
Continent,,,
Africa,1.215738e+09,17759,14.607591
Asia,3.020510e+09,85740,28.385939
Australia/Oceania,4.039107e+07,189,4.679252
Europe,6.794130e+08,155775,229.278789
North America,2.568683e+08,59215,230.526685
South America,4.308076e+08,135506,314.539491


In [77]:
final=merged.sort_values(by='Deaths per million', ascending=False)
final
#alterada a ordem da tabela com base na mortantade por covid,
#em ordem decrescente (4(g))

,Population,Deaths,Deaths per million
Continent,,,
South America,4.308076e+08,135506,314.539491
North America,2.568683e+08,59215,230.526685
Europe,6.794130e+08,155775,229.278789
Asia,3.020510e+09,85740,28.385939
Africa,1.215738e+09,17759,14.607591
Australia/Oceania,4.039107e+07,189,4.679252


In [80]:
final.to_parquet('df_final.parquet', index=True)